Google Colab Integration

In [ ]:

# ============================================================
# Milestone 06B — Colab Cell 1: GPU Environment Verification
# ============================================================

import sys
import platform
import torch

print("GOOGLE COLAB ENVIRONMENT")
print("=" * 55)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Selected device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())

    # Test a small tensor calculation on the GPU
    test_tensor = torch.tensor(
        [1.0, 2.0, 3.0],
        device=DEVICE
    )

    print("GPU tensor result:", (test_tensor * 2).tolist())
    print("GPU computation: PASS")
else:
    print("GPU not available in this runtime.")

print("\nColab environment inspection: PASS")

**Dependency Verification**

In [ ]:

# ============================================================
# Milestone 06B — Colab Cell 2: Dependency Verification
# ============================================================

import importlib.metadata as metadata
import shutil
import platform

# Libraries used in our fraud detection project
required_packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "joblib",
]

optional_packages = [
    "matplotlib",
    "pytest",
]

print("GOOGLE COLAB — DEPENDENCY AUDIT")
print("=" * 55)

missing_required = []

for package in required_packages + optional_packages:
    try:
        version = metadata.version(package)
        print(f"{package}: {version} — INSTALLED")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

        if package in required_packages:
            missing_required.append(package)

print("\nDEVELOPMENT TOOLS")
print("=" * 55)

print("Python:", platform.python_version())
print("Git available:", shutil.which("git") is not None)

print("\nVERIFICATION")
print("=" * 55)

assert not missing_required, (
    f"Missing required libraries: {missing_required}"
)

assert shutil.which("git") is not None, (
    "Git command not available"
)

print("Required dependencies: PASS")
print("Git availability: PASS")
print("Colab dependency audit: PASS")

**Connect Google Colab to GitHub**

In [ ]:

# ============================================================
# Milestone 06B — Colab Cell 3: Connect to Pinned GitHub Source
# ============================================================

from pathlib import Path
import subprocess
import sys
import os

# 1. Define the GitHub repository and verified source commit
REPO_URL = (
    "https://github.com/thekovakgroup/"
    "bank-account-fraud-dl-mlops.git"
)

REPO_DIR = Path("/content/bank-account-fraud-dl-mlops")

EXPECTED_SOURCE_COMMIT = (
    "cf443a546ff655dfcdc01db72d57ae16d022b994"
)

git_env = {
    **os.environ,
    "GIT_TERMINAL_PROMPT": "0",
}

# 2. Clone the repository when no checkout exists
if not REPO_DIR.exists():

    # Full history allows checkout of the historical commit.
    # No checkout is performed until the pinned revision is selected.
    subprocess.run(
        [
            "git",
            "clone",
            "--no-checkout",
            "--branch",
            "main",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
        env=git_env,
    )

    # Check out the precise source revision used for training
    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            "checkout",
            "--detach",
            EXPECTED_SOURCE_COMMIT,
        ],
        check=True,
        env=git_env,
    )

else:
    assert (REPO_DIR / ".git").is_dir(), (
        "Existing directory is not a Git repository."
    )

    print("Existing repository checkout found.")

# 3. Verify the complete source commit BEFORE importing code
actual_commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

assert actual_commit == EXPECTED_SOURCE_COMMIT, (
    f"Unexpected source revision: {actual_commit}. "
    "Stop before importing modules or training."
)

# 4. Verify the required project files
required_files = [
    "src/models/mlp.py",
    "src/training/loops.py",
    "notebooks/05_pytorch_mlp.ipynb",
]

for filename in required_files:
    assert (REPO_DIR / filename).is_file(), (
        f"Missing required file: {filename}"
    )
    print("FOUND:", filename)

# 5. Make the verified source importable
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.models.mlp import FraudMLP

# 6. Verify the original MLP architecture
model = FraudMLP(
    input_dim=55,
    dropout_rate=0.10,
)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

assert parameter_count == 5697

# 7. Report source verification
print("\nGITHUB SOURCE PINNING")
print("=" * 55)

print("Repository:", REPO_DIR)
print("Source commit:", actual_commit)
print("MLP parameters:", parameter_count)
print("Selected device:", DEVICE)

print("\nExact source revision: PASS")
print("Required source modules: PASS")
print("Reusable MLP architecture: PASS")
print("GitHub source pinning: PASS")


**Benchmark CPU vs. Tesla T4 GPU**

In [ ]:

# ============================================================
# Milestone 06B — Colab Cell 4: CPU vs. GPU Benchmark
# ============================================================

import time
import numpy as np
import torch
from torch import nn
from src.models.mlp import FraudMLP

# 1. Configure the benchmark
BENCHMARK_SEED = 42
N_SAMPLES = 8192
N_FEATURES = 55
BATCH_SIZE = 1024
WARMUP_STEPS = 10
TIMED_STEPS = 100

assert torch.cuda.is_available(), "GPU runtime required"

# 2. Create synthetic data (NOT the real fraud dataset)
generator = torch.Generator().manual_seed(BENCHMARK_SEED)

X_synthetic_cpu = torch.randn(
    N_SAMPLES, N_FEATURES, generator=generator
)

y_synthetic_cpu = (
    torch.rand(N_SAMPLES, generator=generator) < 0.015
).float()

print("SYNTHETIC BENCHMARK DATA")
print("Samples:", N_SAMPLES)
print("Features:", N_FEATURES)
print("Positive labels:", int(y_synthetic_cpu.sum()))
print("Batch size:", BATCH_SIZE)

# 3. Save identical initial model weights
torch.manual_seed(BENCHMARK_SEED)

reference_model = FraudMLP(
    input_dim=N_FEATURES,
    dropout_rate=0.10
)

initial_weights = {
    key: tensor.detach().clone()
    for key, tensor in reference_model.state_dict().items()
}


# 4. Define the training benchmark
def benchmark_training(device_name):
    device = torch.device(device_name)

    # Transfer data BEFORE timing
    features = X_synthetic_cpu.to(device)
    labels = y_synthetic_cpu.to(device)

    model = FraudMLP(
        input_dim=N_FEATURES,
        dropout_rate=0.10
    ).to(device)

    model.load_state_dict(initial_weights)
    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    criterion = nn.BCEWithLogitsLoss()

    def training_step(step_number):
        start = (step_number % (N_SAMPLES // BATCH_SIZE)) * BATCH_SIZE
        end = start + BATCH_SIZE

        optimizer.zero_grad(set_to_none=True)

        logits = model(features[start:end])
        loss = criterion(logits, labels[start:end])

        loss.backward()
        optimizer.step()

        return loss

    # Warm up both devices before measuring
    for step in range(WARMUP_STEPS):
        training_step(step)

    if device.type == "cuda":
        torch.cuda.synchronize()

    # Measure training only
    start_time = time.perf_counter()

    for step in range(TIMED_STEPS):
        loss = training_step(step)

    # CUDA operations are asynchronous
    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start_time

    return {
        "device": device_name,
        "seconds": elapsed,
        "steps_per_second": TIMED_STEPS / elapsed,
        "final_loss": float(loss.detach().cpu()),
    }


# 5. Run both benchmarks
cpu_result = benchmark_training("cpu")
gpu_result = benchmark_training("cuda")

# 6. Compare results
speedup = cpu_result["seconds"] / gpu_result["seconds"]

print("\nCPU vs GPU — TRAINING BENCHMARK")
print("=" * 55)

for result in [cpu_result, gpu_result]:
    print(
        f"{result['device'].upper()}: "
        f"{result['seconds']:.3f} seconds | "
        f"{result['steps_per_second']:.2f} steps/sec"
    )

print(f"\nGPU speedup factor: {speedup:.2f}x")

# 7. Verify benchmark results
assert np.isfinite(cpu_result["final_loss"])
assert np.isfinite(gpu_result["final_loss"])
assert cpu_result["seconds"] > 0
assert gpu_result["seconds"] > 0

print("\nSynthetic training benchmark: PASS")
print("CPU execution: PASS")
print("GPU execution: PASS")
print("Real fraud dataset accessed: NO")

**Benchmark Reliability**

In [ ]:

# ============================================================
# Milestone 06B — Colab Cell 5: Repeated CPU/GPU Benchmark
# ============================================================

import statistics
import pandas as pd

N_TRIALS = 5
benchmark_records = []

# 1. Repeat the benchmark on both devices
for trial in range(1, N_TRIALS + 1):

    # Alternate the order to reduce systematic order effects
    devices = (
        ["cpu", "cuda"] if trial % 2 else ["cuda", "cpu"]
    )

    for device_name in devices:
        result = benchmark_training(device_name)

        benchmark_records.append({
            "trial": trial,
            "device": device_name,
            "seconds": result["seconds"],
            "steps_per_second": result["steps_per_second"],
        })

# 2. Create a results table
benchmark_df = pd.DataFrame(benchmark_records)

# 3. Calculate median performance for each device
cpu_median = statistics.median(
    benchmark_df.loc[
        benchmark_df["device"] == "cpu", "seconds"
    ]
)

gpu_median = statistics.median(
    benchmark_df.loc[
        benchmark_df["device"] == "cuda", "seconds"
    ]
)

median_speedup = cpu_median / gpu_median

# 4. Verify results
assert len(benchmark_df) == 2 * N_TRIALS
assert benchmark_df.groupby("device").size().eq(N_TRIALS).all()
assert cpu_median > 0
assert gpu_median > 0
assert np.isfinite(median_speedup)

# 5. Display results
print("REPEATED CPU/GPU BENCHMARK")
print("=" * 55)

display(
    benchmark_df.pivot(
        index="trial",
        columns="device",
        values="seconds"
    ).round(4)
)

print("\nMEDIAN PERFORMANCE")
print("=" * 55)
print(f"CPU median: {cpu_median:.4f} seconds")
print(f"GPU median: {gpu_median:.4f} seconds")
print(f"GPU speedup: {median_speedup:.2f}x")

print("\nFive trials per device: PASS")
print("Benchmark reliability check: PASS")
print("Real fraud dataset accessed: NO")

**Check Preprocessing Compatibility**

In [ ]:

# ============================================================
# Milestone 06B — Colab Cell 6: Source & Compatibility Audit
# ============================================================

import platform
import sklearn
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# Versions from the verified local Milestone 05 experiment
LOCAL_REFERENCE = {
    "python": "3.11.9",
    "pytorch": "2.14.1+cpu",
    "numpy": "2.4.6",
    "pandas": "3.0.6",
    "scikit_learn": "1.9.1",
}

COLAB_ENVIRONMENT = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
}

print("ENVIRONMENT COMPATIBILITY")
print("=" * 65)

for package, original_version in LOCAL_REFERENCE.items():
    colab_version = COLAB_ENVIRONMENT[package]

    print(
        f"{package:15} "
        f"Local: {original_version:15} "
        f"Colab: {colab_version}"
    )

# Inspect source code already cloned from GitHub
source_dir = REPO_DIR / "src"

assert source_dir.is_dir()

source_modules = sorted(
    path.relative_to(REPO_DIR).as_posix()
    for path in source_dir.rglob("*.py")
)

print("\nAVAILABLE PROJECT MODULES")
print("=" * 65)

for module in source_modules:
    print(module)

# Detect potential sklearn serialization incompatibility
sklearn_versions_match = (
    LOCAL_REFERENCE["scikit_learn"]
    == COLAB_ENVIRONMENT["scikit_learn"]
)

print("\nPREPROCESSING COMPATIBILITY")
print("=" * 65)

print("Scikit-learn versions match:", sklearn_versions_match)

if not sklearn_versions_match:
    print("Saved local joblib pipeline: DO NOT LOAD IN COLAB")
    print("Preferred approach: verified portable feature arrays")

assert "src/models/mlp.py" in source_modules
assert "src/training/loops.py" in source_modules

print("\nSource modules verified: PASS")
print("Compatibility differences identified: PASS")
print("Real fraud dataset accessed: NO")

**Verify File Integrity in Google Colab**

In [ ]:

# ============================================================
# Milestone 06B — Colab Cell 7: Uploaded File Integrity
# ============================================================

from pathlib import Path
import hashlib
import json

# 1. Locate the uploaded files
COLAB_DATA_DIR = Path("/content")

ARRAYS_PATH = (
    COLAB_DATA_DIR / "colab_dev_arrays_v01.npz"
)

MANIFEST_PATH = (
    COLAB_DATA_DIR / "colab_dev_manifest_v01.json"
)

# 2. Confirm both files exist
assert ARRAYS_PATH.is_file(), (
    f"Missing data archive: {ARRAYS_PATH}"
)

assert MANIFEST_PATH.is_file(), (
    f"Missing manifest: {MANIFEST_PATH}"
)

# 3. Load the small JSON manifest
with MANIFEST_PATH.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

# 4. Recalculate uploaded archive checksum
sha256 = hashlib.sha256()

with ARRAYS_PATH.open("rb") as file:
    for chunk in iter(
        lambda: file.read(1024 * 1024),
        b""
    ):
        sha256.update(chunk)

uploaded_sha256 = sha256.hexdigest()

# Original SHA-256 verified in local VS Code
EXPECTED_SHA256 = (
    "b0924b639b9f38eceb2db413f7fa710b94d"
    "bc71d1223ab95a7db8f106be05fd5"
)

# 5. Compare independent integrity records
assert uploaded_sha256 == EXPECTED_SHA256, (
    "Uploaded archive does not match local export!"
)

assert manifest["archive_sha256"] == EXPECTED_SHA256
assert manifest["archive_name"] == ARRAYS_PATH.name

# 6. Verify declared dataset boundaries
assert manifest["training"]["months"] == list(range(6))
assert manifest["validation"]["months"] == [6]
assert manifest["holdout"]["status"] == "SEALED"
assert manifest["holdout"]["included_in_archive"] is False

# 7. Display verification
print("GOOGLE COLAB — FILE INTEGRITY")
print("=" * 60)

print("Data archive found:", ARRAYS_PATH.is_file())
print("Manifest found:", MANIFEST_PATH.is_file())
print(
    "Archive size (MiB):",
    round(ARRAYS_PATH.stat().st_size / (1024 ** 2), 2)
)

print("\nUploaded SHA-256:", uploaded_sha256)

print("\nChecksum matches local export: PASS")
print("Manifest checksum matches: PASS")
print("Temporal split declarations: PASS")
print("Test status: SEALED")
print("Colab upload integrity: PASS")

Diagnose hash Mismatch

In [ ]:

# ============================================================
# Step 06.23B.1 — Diagnose Uploaded Archive
# READ-ONLY: No file modifications
# ============================================================

print("COLAB ARCHIVE DIAGNOSTIC")
print("=" * 60)

print("Archive path:", ARRAYS_PATH)
print("Archive exists:", ARRAYS_PATH.is_file())
print("Archive size (MiB):", round(
    ARRAYS_PATH.stat().st_size / (1024 ** 2), 2
))

print("\nSHA-256 COMPARISON")
print("=" * 60)

print("Local verified SHA-256:")
print("b0924b639b9f38eceb2db413f7fa710b94dbc71d1223ab95a7db8f106be05fd5")

print("\nExpected SHA-256 in code:")
print(EXPECTED_SHA256)

print("\nUploaded file SHA-256:")
print(uploaded_sha256)

print("\nManifest SHA-256:")
print(manifest["archive_sha256"])

print("\nDIAGNOSTIC RESULTS")
print("=" * 60)

print(
    "Uploaded matches manifest:",
    uploaded_sha256 == manifest["archive_sha256"]
)

print(
    "Code expectation matches manifest:",
    EXPECTED_SHA256 == manifest["archive_sha256"]
)

print(
    "Available NPZ archives:",
    [p.name for p in sorted(
        Path("/content").glob("*.npz")
    )]
)

print("\nDiagnostic completed: PASS")

Check the Actual File Hash

In [ ]:

from pathlib import Path
import hashlib
import json

archive = Path("/content/colab_dev_arrays_v01.npz")
manifest_file = Path("/content/colab_dev_manifest_v01.json")

assert archive.is_file()
assert manifest_file.is_file()

manifest = json.loads(manifest_file.read_text())

actual_hash = hashlib.sha256(
    archive.read_bytes()
).hexdigest()

original_hash = (
    "b0924b639b9f38eceb2db413f7fa710b94dbc71d1223ab95a7db8f106be05fd5"
)

print("ARCHIVE INTEGRITY DIAGNOSTIC")
print("=" * 55)

print("Archive size (MiB):", round(
    archive.stat().st_size / (1024 ** 2), 2
))

print("\nOriginal local SHA-256:")
print(original_hash)

print("\nUploaded Colab SHA-256:")
print(actual_hash)

print("\nManifest SHA-256:")
print(manifest["archive_sha256"])

print("\nVERIFICATION")
print("Uploaded matches original:", actual_hash == original_hash)
print("Uploaded matches manifest:", actual_hash == manifest["archive_sha256"])
print("Manifest matches original:", manifest["archive_sha256"] == original_hash)

**Load and Validate Our Development Dataset**

In [ ]:

# ============================================================
# Milestone 06B — Colab Cell 8: Load & Validate Development Data
# ============================================================

import numpy as np
import json
import hashlib
from pathlib import Path

# 1. Define uploaded file locations
DATA_DIR = Path("/content")

ARRAYS_PATH = DATA_DIR / "colab_dev_arrays_v01.npz"
MANIFEST_PATH = DATA_DIR / "colab_dev_manifest_v01.json"

# Handle Colab runtime disconnections
if not ARRAYS_PATH.is_file() or not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "Development files are missing from /content/. "
        "The Colab runtime may have restarted. "
        "Re-upload the verified NPZ archive and JSON manifest."
    )

# 2. Load the manifest
with MANIFEST_PATH.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

# 3. Reverify archive integrity
expected_sha256 = (
    "b0924b639b9f38eceb2db413f7fa710b94dbc71d1223ab95a7db8f106be05fd5"
)

hash_object = hashlib.sha256()

with ARRAYS_PATH.open("rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        hash_object.update(chunk)

actual_sha256 = hash_object.hexdigest()

assert actual_sha256 == expected_sha256
assert manifest["archive_sha256"] == actual_sha256

print("Archive integrity: PASS")

# 4. Load portable NumPy arrays
with np.load(ARRAYS_PATH, allow_pickle=False) as archive:

    expected_keys = {
        "X_train",
        "y_train",
        "X_validation",
        "y_validation",
        "feature_names",
    }

    assert set(archive.files) == expected_keys

    colab_X_train = archive["X_train"]
    colab_y_train = archive["y_train"]

    colab_X_validation = archive["X_validation"]
    colab_y_validation = archive["y_validation"]

    colab_feature_names = archive["feature_names"].tolist()

# 5. Verify matrix dimensions
assert colab_X_train.shape == (794989, 55)
assert colab_X_validation.shape == (108168, 55)

assert colab_y_train.shape == (794989,)
assert colab_y_validation.shape == (108168,)

# 6. Verify PyTorch-compatible dtypes
assert colab_X_train.dtype == np.float32
assert colab_X_validation.dtype == np.float32

assert colab_y_train.dtype == np.float32
assert colab_y_validation.dtype == np.float32

# 7. Verify numerical integrity
assert np.isfinite(colab_X_train).all()
assert np.isfinite(colab_X_validation).all()

assert np.isfinite(colab_y_train).all()
assert np.isfinite(colab_y_validation).all()

# 8. Verify fraud labels
assert set(np.unique(colab_y_train)) == {0.0, 1.0}
assert set(np.unique(colab_y_validation)) == {0.0, 1.0}

train_fraud_count = int(colab_y_train.sum())
validation_fraud_count = int(colab_y_validation.sum())

assert train_fraud_count == 8151
assert validation_fraud_count == 1450

# 9. Verify feature order and temporal declarations
assert colab_feature_names == manifest["features"]["ordered_names"]
assert len(colab_feature_names) == 55

assert manifest["training"]["months"] == list(range(6))
assert manifest["validation"]["months"] == [6]
assert manifest["holdout"]["status"] == "SEALED"
assert manifest["holdout"]["included_in_archive"] is False

# 10. Display verification results
print("\nGOOGLE COLAB — DEVELOPMENT DATA VALIDATION")
print("=" * 60)

print("Training features:", colab_X_train.shape)
print("Validation features:", colab_X_validation.shape)

print("Training labels:", colab_y_train.shape)
print("Validation labels:", colab_y_validation.shape)

print("\nTraining fraud cases:", train_fraud_count)
print("Validation fraud cases:", validation_fraud_count)

print(
    "Training fraud rate:",
    f"{colab_y_train.mean():.4%}"
)

print(
    "Validation fraud rate:",
    f"{colab_y_validation.mean():.4%}"
)

print("\nFeature dtype:", colab_X_train.dtype)
print("Label dtype:", colab_y_train.dtype)
print("Feature count:", len(colab_feature_names))

print("\nArray shapes: PASS")
print("Feature order: PASS")
print("Float32 compatibility: PASS")
print("Binary labels: PASS")
print("Class distributions: PASS")
print("Finite numerical values: PASS")
print("Test split declarations: SEALED")
print("Development data validation: PASS")

In [ ]:

# Step 06.24C — Diagnose Uploaded File Integrity
# Read-only: does not modify or load training arrays

from pathlib import Path
import hashlib
import json
import time

archive = Path("/content/colab_dev_arrays_v01.npz")
manifest_path = Path("/content/colab_dev_manifest_v01.json")

assert archive.is_file()
assert manifest_path.is_file()

manifest = json.loads(manifest_path.read_text())

expected_hash = (
    "b0924b639b9f38eceb2db413f7fa710b94dbc71d1223ab95a7db8f106be05fd5"
)

def inspect_file(path):
    before = path.stat()
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    after = path.stat()

    return (
        digest.hexdigest(),
        after.st_size,
        before.st_size == after.st_size
        and before.st_mtime_ns == after.st_mtime_ns
    )

# Measure twice to detect a changing upload
first_hash, first_size, first_stable = inspect_file(archive)

time.sleep(5)

second_hash, second_size, second_stable = inspect_file(archive)

print("UPLOAD INTEGRITY DIAGNOSTIC")
print("=" * 55)

print("First SHA-256:", first_hash)
print("Second SHA-256:", second_hash)
print("Expected SHA-256:", expected_hash)
print("Manifest SHA-256:", manifest["archive_sha256"])

print("\nFile size (MiB):", round(second_size / 1024**2, 2))
print("Same hash in both checks:", first_hash == second_hash)
print("File unchanged during checks:", first_stable and second_stable)
print("Matches original export:", second_hash == expected_hash)
print("Matches manifest:", second_hash == manifest["archive_sha256"])

Restore the GitHub Repository

In [ ]:

# ============================================================
# Milestone 06B — Step 06.25A: Restore Pinned GitHub Source
# ============================================================

from pathlib import Path
import subprocess
import sys
import os
import torch

# 1. Define repository and original training source
REPO_URL = (
    "https://github.com/thekovakgroup/"
    "bank-account-fraud-dl-mlops.git"
)

REPO_DIR = Path("/content/bank-account-fraud-dl-mlops")

EXPECTED_SOURCE_COMMIT = (
    "cf443a546ff655dfcdc01db72d57ae16d022b994"
)

git_env = {
    **os.environ,
    "GIT_TERMINAL_PROMPT": "0",
}

# 2. Restore repository if the runtime was restarted
if not REPO_DIR.exists():

    # Retrieve history so the original commit can be checked out
    subprocess.run(
        [
            "git",
            "clone",
            "--no-checkout",
            "--branch",
            "main",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
        env=git_env,
    )

    # Select the exact historical revision
    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            "checkout",
            "--detach",
            EXPECTED_SOURCE_COMMIT,
        ],
        check=True,
        env=git_env,
    )

else:
    assert (REPO_DIR / ".git").is_dir(), (
        "Existing directory is not a Git repository."
    )

    print("Existing repository checkout found.")

# 3. Verify the FULL source revision before importing
actual_commit = subprocess.check_output(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

assert actual_commit == EXPECTED_SOURCE_COMMIT, (
    f"Unexpected source revision: {actual_commit}. "
    "Stop before importing modules or training."
)

# 4. Verify required source modules
required_files = [
    "src/models/mlp.py",
    "src/training/loops.py",
]

for filename in required_files:
    assert (REPO_DIR / filename).is_file(), (
        f"Missing module: {filename}"
    )
    print("FOUND:", filename)

# 5. Restore Python imports only after revision verification
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.models.mlp import FraudMLP

# 6. Verify original neural-network architecture
model = FraudMLP(
    input_dim=55,
    dropout_rate=0.10,
)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

assert parameter_count == 5697

# 7. Verify the restored Colab environment
assert torch.cuda.is_available(), (
    "CUDA GPU unavailable. Restore GPU runtime."
)

assert "colab_X_train" in globals(), (
    "Training arrays unavailable. Reload development data."
)

assert "colab_X_validation" in globals(), (
    "Validation arrays unavailable. Reload development data."
)

assert colab_X_train.shape == (794989, 55)
assert colab_X_validation.shape == (108168, 55)

# 8. Display verification results
print("\nPINNED GITHUB REPOSITORY RECOVERY")
print("=" * 60)

print("Source commit:", actual_commit)
print("Trainable parameters:", parameter_count)
print("GPU:", torch.cuda.get_device_name(0))
print("Training data:", colab_X_train.shape)
print("Validation data:", colab_X_validation.shape)

print("\nExact source revision: PASS")
print("Required source modules: PASS")
print("Original MLP architecture: PASS")
print("Development arrays available: PASS")
print("Month-7 test set: SEALED")
print("Runtime recovery: PASS")


**Verify GPU and GitHub After the Runtime Restart**

In [ ]:

# ============================================================
# Milestone 06B — Colab Cell 9: Runtime Readiness Check
# ============================================================

from pathlib import Path
import subprocess
import torch

# 1. Verify GPU availability
gpu_ready = torch.cuda.is_available()

print("GOOGLE COLAB — RUNTIME READINESS")
print("=" * 60)

print("CUDA available:", gpu_ready)

if gpu_ready:
    print("GPU:", torch.cuda.get_device_name(0))
    print("PyTorch:", torch.__version__)

# 2. Check the cloned GitHub repository
repo_path = Path("/content/bank-account-fraud-dl-mlops")

required_modules = [
    repo_path / "src/models/mlp.py",
    repo_path / "src/training/loops.py",
]

repo_ready = all(path.is_file() for path in required_modules)

print("\nGITHUB REPOSITORY")
print("=" * 60)
print("Repository exists:", repo_path.is_dir())
print("Required modules available:", repo_ready)

if repo_ready:
    result = subprocess.run(
        ["git", "-C", str(repo_path), "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True,
        check=False,
    )
    print("Git commit:", result.stdout.strip() or "Unavailable")

# 3. Verify development arrays remain in memory
array_names = [
    "colab_X_train",
    "colab_y_train",
    "colab_X_validation",
    "colab_y_validation",
    "colab_feature_names",
]

arrays_ready = all(
    name in globals() for name in array_names
)

print("\nDEVELOPMENT DATA")
print("=" * 60)
print("Development arrays available:", arrays_ready)

if arrays_ready:
    print("Training shape:", colab_X_train.shape)
    print("Validation shape:", colab_X_validation.shape)

# 4. Report overall readiness
print("\nFINAL READINESS")
print("=" * 60)

if gpu_ready and repo_ready and arrays_ready:
    print("Runtime readiness: PASS")
    print("Ready for PyTorch DataLoaders: YES")
else:
    print("Runtime readiness: RESTORE NEEDED")
    print("Ready for PyTorch DataLoaders: NO")

**Create PyTorch Tensors and DataLoaders**

In [ ]:

# ============================================================
# Milestone 06B — Cell 10: PyTorch Tensors and DataLoaders
# CORRECTED VERSION
# ============================================================

import torch
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
BATCH_SIZE = 1024

# 1. Convert validated NumPy arrays into PyTorch tensors
# Keep labels one-dimensional to match FraudMLP logits.

m06_X_train_tensor = torch.from_numpy(
    colab_X_train
)

m06_y_train_tensor = torch.from_numpy(
    colab_y_train
).reshape(-1)

m06_X_validation_tensor = torch.from_numpy(
    colab_X_validation
)

m06_y_validation_tensor = torch.from_numpy(
    colab_y_validation
).reshape(-1)

# 2. Verify tensor shapes
assert m06_X_train_tensor.shape == (794989, 55)
assert m06_y_train_tensor.shape == (794989,)

assert m06_X_validation_tensor.shape == (108168, 55)
assert m06_y_validation_tensor.shape == (108168,)

# 3. Verify tensor data types
for tensor in [
    m06_X_train_tensor,
    m06_y_train_tensor,
    m06_X_validation_tensor,
    m06_y_validation_tensor,
]:
    assert tensor.dtype == torch.float32

# 4. Create training and validation datasets
m06_train_dataset = TensorDataset(
    m06_X_train_tensor,
    m06_y_train_tensor,
)

m06_validation_dataset = TensorDataset(
    m06_X_validation_tensor,
    m06_y_validation_tensor,
)

# 5. Create reproducible training DataLoader
m06_train_loader = DataLoader(
    m06_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

# 6. Create validation DataLoader
m06_validation_loader = DataLoader(
    m06_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

# 7. Verify batch dimensions
batch_X, batch_y = next(iter(m06_validation_loader))

assert batch_X.shape == (BATCH_SIZE, 55)
assert batch_y.shape == (BATCH_SIZE,)

# 8. Verify dataset and batch counts
assert len(m06_train_dataset) == 794989
assert len(m06_validation_dataset) == 108168

assert len(m06_train_loader) == 777
assert len(m06_validation_loader) == 106

# 9. Display validation results
print("PYTORCH DATALOADER VALIDATION")
print("=" * 55)

print("Training records:", len(m06_train_dataset))
print("Validation records:", len(m06_validation_dataset))

print("\nTraining batches:", len(m06_train_loader))
print("Validation batches:", len(m06_validation_loader))

print("\nBatch feature shape:", tuple(batch_X.shape))
print("Batch label shape:", tuple(batch_y.shape))

print("\nTensor dtype:", batch_X.dtype)
print("Training shuffled:", True)
print("Validation shuffled:", False)
print("Batch size:", BATCH_SIZE)
print("Pinned memory:", torch.cuda.is_available())

print("\nTensor conversion: PASS")
print("DataLoader construction: PASS")
print("Batch dimensions: PASS")
print("Month-7 test set: SEALED")
print("Step 06.26: PASS")

**Define and Verify Class-Weighted Loss**

In [ ]:

# ============================================================
# Milestone 06B — Cell 11: Class-Weighted Loss Verification
# ============================================================

import torch
from torch import nn
from torch.nn import functional as F

# 1. Calculate class counts using training labels ONLY
m06_positive_count = int(m06_y_train_tensor.sum().item())
m06_total_count = m06_y_train_tensor.numel()
m06_negative_count = m06_total_count - m06_positive_count

assert m06_positive_count == 8151
assert m06_negative_count == 786838

# 2. Calculate positive class weight
m06_pos_weight_value = (
    m06_negative_count / m06_positive_count
)

# 3. Create both loss functions on the available device
m06_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

m06_pos_weight_tensor = torch.tensor(
    [m06_pos_weight_value],
    dtype=torch.float32,
    device=m06_device,
)

m06_unweighted_loss = nn.BCEWithLogitsLoss().to(m06_device)

m06_weighted_loss = nn.BCEWithLogitsLoss(
    pos_weight=m06_pos_weight_tensor
).to(m06_device)

# 4. Test with two synthetic examples
# Logit 0 = predicted probability 0.5
test_logits = torch.zeros(
    (2, 1), dtype=torch.float32, device=m06_device
)

# One legitimate example, one fraud example
test_targets = torch.tensor(
    [[0.0], [1.0]],
    dtype=torch.float32,
    device=m06_device,
)

with torch.no_grad():
    ordinary_losses = F.binary_cross_entropy_with_logits(
        test_logits,
        test_targets,
        reduction="none",
    )

    weighted_losses = F.binary_cross_entropy_with_logits(
        test_logits,
        test_targets,
        pos_weight=m06_pos_weight_tensor,
        reduction="none",
    )

# 5. Verify mathematical behavior
assert torch.allclose(
    weighted_losses[0], ordinary_losses[0]
), "Negative-class loss must remain unchanged."

assert torch.allclose(
    weighted_losses[1] / ordinary_losses[1],
    m06_pos_weight_tensor,
    rtol=1e-5,
), "Fraud loss must scale by pos_weight."

assert torch.isfinite(weighted_losses).all()

print("CLASS-WEIGHTED LOSS VERIFICATION")
print("=" * 55)

print("Training fraud cases:", m06_positive_count)
print("Training legitimate cases:", m06_negative_count)
print("Positive class weight:", round(m06_pos_weight_value, 4))
print("Device:", m06_device)

print("\nSYNTHETIC LOSS CHECK")
print("=" * 55)

print(
    "Legitimate loss — standard:",
    round(ordinary_losses[0].item(), 4)
)
print(
    "Legitimate loss — weighted:",
    round(weighted_losses[0].item(), 4)
)

print(
    "Fraud loss — standard:",
    round(ordinary_losses[1].item(), 4)
)
print(
    "Fraud loss — weighted:",
    round(weighted_losses[1].item(), 4)
)

print("\nTraining-only weight calculation: PASS")
print("Legitimate loss unchanged: PASS")
print("Fraud loss increased correctly: PASS")
print("Numerical stability: PASS")
print("Month-7 test set: SEALED")
print("Step 06.27: PASS")

**Initialize the Unweighted GPU Reference Model**

In [ ]:

# ============================================================
# Milestone 06B — Cell 12: Unweighted GPU Reference Setup
# ============================================================

import random
import numpy as np
import torch
from torch import nn

from src.models.mlp import FraudMLP

# 1. Match our Milestone 05 reference configuration
M06_REFERENCE_SEED = 42
M06_REFERENCE_EPOCHS = 5
M06_REFERENCE_LR = 0.001
M06_REFERENCE_BATCH_SIZE = 1024

# 2. Set random seeds before model initialization
random.seed(M06_REFERENCE_SEED)
np.random.seed(M06_REFERENCE_SEED)
torch.manual_seed(M06_REFERENCE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(M06_REFERENCE_SEED)

# 3. Require the verified GPU
assert torch.cuda.is_available(), "CUDA GPU unavailable."

m06_reference_device = torch.device("cuda")

# 4. Initialize the original MLP architecture
m06_reference_model = FraudMLP(
    input_dim=55,
    dropout_rate=0.10,
).to(m06_reference_device)

# 5. Use the original unweighted loss
m06_reference_criterion = nn.BCEWithLogitsLoss()

assert m06_reference_criterion.pos_weight is None

# 6. Use the original optimizer configuration
m06_reference_optimizer = torch.optim.Adam(
    m06_reference_model.parameters(),
    lr=M06_REFERENCE_LR,
)

# 7. Verify the architecture
m06_parameter_count = sum(
    p.numel()
    for p in m06_reference_model.parameters()
)

assert m06_parameter_count == 5697

assert m06_train_loader.batch_size == M06_REFERENCE_BATCH_SIZE
assert m06_validation_loader.batch_size == M06_REFERENCE_BATCH_SIZE

# 8. Run a synthetic forward-pass test
# This does NOT train or modify model weights.
m06_reference_model.eval()

with torch.inference_mode():
    sample_inputs = torch.zeros(
        (4, 55),
        dtype=torch.float32,
        device=m06_reference_device,
    )
    sample_logits = m06_reference_model(sample_inputs)

assert sample_logits.shape == (4,)
assert torch.isfinite(sample_logits).all()

# Restore training mode for the next step
m06_reference_model.train()

# 9. Report configuration
print("UNWEIGHTED GPU REFERENCE — SETUP")
print("=" * 55)

print("Device:", m06_reference_device)
print("GPU:", torch.cuda.get_device_name(0))
print("Random seed:", M06_REFERENCE_SEED)
print("Input features:", 55)
print("Trainable parameters:", m06_parameter_count)

print("\nTRAINING CONFIGURATION")
print("=" * 55)

print("Optimizer: Adam")
print("Learning rate:", M06_REFERENCE_LR)
print("Loss: BCEWithLogitsLoss")
print("Class weighting: DISABLED")
print("Batch size:", M06_REFERENCE_BATCH_SIZE)
print("Epochs:", M06_REFERENCE_EPOCHS)

print("\nFORWARD PASS")
print("=" * 55)

print("Input shape:", tuple(sample_inputs.shape))
print("Output logits shape:", tuple(sample_logits.shape))
print("Finite logits:", bool(torch.isfinite(sample_logits).all()))

print("\nArchitecture verification: PASS")
print("Unweighted loss verification: PASS")
print("Optimizer initialization: PASS")
print("Forward-pass verification: PASS")
print("Training performed: NO")
print("Month-7 test set: SEALED")
print("Step 06.28A: PASS")

In [ ]:

# Step 06.28A.1 — Diagnose MLP Output Dimensions
# Read-only inspection: NO training

import inspect
import torch

print("FRAUD MLP — OUTPUT DIAGNOSTIC")
print("=" * 55)

print("Output type:", type(sample_logits).__name__)

if isinstance(sample_logits, torch.Tensor):
    print("Actual output shape:", tuple(sample_logits.shape))
    print("Expected output shape:", (4, 1))
    print("Number of output values:", sample_logits.numel())
    print("Output dtype:", sample_logits.dtype)
    print("All values finite:", bool(torch.isfinite(sample_logits).all()))

print("\nLABEL DIMENSIONS")
print("=" * 55)
print("Training labels:", tuple(m06_y_train_tensor.shape))
print("Validation labels:", tuple(m06_y_validation_tensor.shape))

print("\nFRAUD MLP FORWARD METHOD")
print("=" * 55)
print(inspect.getsource(FraudMLP.forward))

print("\nDiagnostic completed: PASS")
print("Training performed: NO")

**Train the Unweighted GPU Reference Model**

In [ ]:

# ============================================================
# Milestone 06B — Cell 13: Train Unweighted GPU Reference
# ============================================================

import time
import numpy as np
import torch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

# 1. Prevent accidentally training the same model twice
if globals().get("m06_reference_run_started", False):
    raise RuntimeError(
        "This reference run has already started. "
        "Do not repeat training without resetting the model "
        "and optimizer. Review results first."
    )

m06_reference_run_started = True

m06_reference_history = []

print("UNWEIGHTED GPU REFERENCE — TRAINING")
print("=" * 60)

print("GPU:", torch.cuda.get_device_name(0))
print("Epochs:", M06_REFERENCE_EPOCHS)
print("Training batches:", len(m06_train_loader))
print("Validation batches:", len(m06_validation_loader))
print("Loss: Unweighted BCEWithLogitsLoss")
print("Month-7 test set: SEALED")
print()

# 2. Train and validate for exactly five epochs
for epoch in range(1, M06_REFERENCE_EPOCHS + 1):

    torch.cuda.synchronize()
    epoch_start = time.perf_counter()

    # ----------------------------
    # TRAINING
    # ----------------------------
    m06_reference_model.train()

    training_loss_total = 0.0
    training_records = 0

    for features, labels in m06_train_loader:

        features = features.to(
            m06_reference_device,
            non_blocking=True,
        )

        labels = labels.to(
            m06_reference_device,
            non_blocking=True,
        )

        m06_reference_optimizer.zero_grad(set_to_none=True)

        logits = m06_reference_model(features)

        assert logits.shape == labels.shape

        loss = m06_reference_criterion(logits, labels)

        loss.backward()
        m06_reference_optimizer.step()

        batch_size = labels.shape[0]

        training_loss_total += loss.item() * batch_size
        training_records += batch_size

    average_training_loss = (
        training_loss_total / training_records
    )

    assert training_records == 794989

    # ----------------------------
    # VALIDATION
    # ----------------------------
    m06_reference_model.eval()

    validation_loss_total = 0.0
    validation_records = 0

    validation_labels = []
    validation_scores = []

    with torch.inference_mode():

        for features, labels in m06_validation_loader:

            features = features.to(
                m06_reference_device,
                non_blocking=True,
            )

            labels = labels.to(
                m06_reference_device,
                non_blocking=True,
            )

            logits = m06_reference_model(features)

            assert logits.shape == labels.shape

            loss = m06_reference_criterion(logits, labels)

            batch_size = labels.shape[0]

            validation_loss_total += loss.item() * batch_size
            validation_records += batch_size

            probabilities = torch.sigmoid(logits)

            validation_labels.append(
                labels.cpu().numpy()
            )

            validation_scores.append(
                probabilities.cpu().numpy()
            )

    assert validation_records == 108168

    y_true = np.concatenate(validation_labels)
    y_score = np.concatenate(validation_scores)

    assert y_true.shape == (108168,)
    assert y_score.shape == (108168,)
    assert int(y_true.sum()) == 1450
    assert np.isfinite(y_score).all()

    average_validation_loss = (
        validation_loss_total / validation_records
    )

    validation_pr_auc = average_precision_score(
        y_true,
        y_score,
    )

    validation_roc_auc = roc_auc_score(
        y_true,
        y_score,
    )

    torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - epoch_start

    # ----------------------------
    # RECORD RESULTS
    # ----------------------------
    epoch_result = {
        "epoch": epoch,
        "train_loss": float(average_training_loss),
        "val_loss": float(average_validation_loss),
        "val_pr_auc": float(validation_pr_auc),
        "val_roc_auc": float(validation_roc_auc),
        "epoch_seconds": float(epoch_seconds),
    }

    m06_reference_history.append(epoch_result)

    print(
        f"Epoch {epoch}/{M06_REFERENCE_EPOCHS} | "
        f"Train Loss: {average_training_loss:.6f} | "
        f"Val Loss: {average_validation_loss:.6f} | "
        f"PR-AUC: {validation_pr_auc:.6f} | "
        f"ROC-AUC: {validation_roc_auc:.6f} | "
        f"Time: {epoch_seconds:.2f}s",
        flush=True,
    )

# 3. Final fixed-epoch reference
assert len(m06_reference_history) == 5

m06_reference_final = m06_reference_history[-1]

m06_reference_total_seconds = sum(
    result["epoch_seconds"]
    for result in m06_reference_history
)

# 4. Compare against Milestone 05 CPU reference
M05_REFERENCE_PR_AUC = 0.15746859566687155

pr_auc_difference = (
    m06_reference_final["val_pr_auc"]
    - M05_REFERENCE_PR_AUC
)

print("\nFINAL GPU REFERENCE RESULTS")
print("=" * 60)

print(
    "GPU validation PR-AUC:",
    round(m06_reference_final["val_pr_auc"], 6)
)

print(
    "GPU validation ROC-AUC:",
    round(m06_reference_final["val_roc_auc"], 6)
)

print(
    "M05 CPU reference PR-AUC:",
    round(M05_REFERENCE_PR_AUC, 6)
)

print(
    "PR-AUC difference (GPU - CPU):",
    round(pr_auc_difference, 6)
)

print(
    "Total training + validation time (seconds):",
    round(m06_reference_total_seconds, 2)
)

print("\nFixed five-epoch reference: PASS")
print("Validation metrics calculated: PASS")
print("Class weighting: DISABLED")
print("Month-7 test set: SEALED")
print("Step 06.28B: PASS")

**Save the Trained Model Before Continuing**

In [ ]:

# ============================================================
# Milestone 06B — Cell 14: Save GPU Reference Artifacts
# ============================================================

from pathlib import Path
from google.colab import files
import torch
import json
import hashlib
import zipfile

# 1. Confirm training finished successfully
assert m06_reference_run_started is True
assert len(m06_reference_history) == 5
assert m06_reference_final["epoch"] == 5

# 2. Create a versioned export directory
export_dir = Path("/content/m06_reference_export_v01")
zip_path = Path("/content/m06_unweighted_gpu_reference_v01.zip")

assert not zip_path.exists(), (
    "Export ZIP already exists. Check it before creating another."
)

export_dir.mkdir(parents=True, exist_ok=True)

weights_path = export_dir / "m06_unweighted_gpu_reference_v01.pt"
history_path = export_dir / "m06_unweighted_gpu_history_v01.json"
manifest_path = export_dir / "m06_unweighted_gpu_manifest_v01.json"

# 3. Save weights as CPU tensors for portability
# Store state_dict only, not the entire Python model object.
cpu_weights = {
    name: tensor.detach().cpu().clone()
    for name, tensor in m06_reference_model.state_dict().items()
}

torch.save(cpu_weights, weights_path)

# 4. Verify saved weights can be safely loaded
loaded_weights = torch.load(
    weights_path,
    map_location="cpu",
    weights_only=True,
)

assert loaded_weights.keys() == cpu_weights.keys()

for name in cpu_weights:
    assert torch.equal(
        loaded_weights[name],
        cpu_weights[name],
    )

# 5. Save all five epoch results
history_document = {
    "experiment_id": "m06_gpu_reference_unweighted_v01",
    "source_commit": "cf443a5",
    "architecture": "FraudMLP",
    "feature_count": 55,
    "parameter_count": 5697,
    "seed": 42,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "batch_size": 1024,
    "epochs": 5,
    "loss_function": "BCEWithLogitsLoss",
    "class_weighting": False,
    "train_months": [0, 1, 2, 3, 4, 5],
    "validation_months": [6],
    "test_status": "SEALED",
    "device": str(m06_reference_device),
    "gpu": torch.cuda.get_device_name(0),
    "pytorch_version": torch.__version__,
    "epoch_history": m06_reference_history,
    "final_validation": m06_reference_final,
    "total_seconds": m06_reference_total_seconds,
}

history_path.write_text(
    json.dumps(history_document, indent=2, allow_nan=False),
    encoding="utf-8",
)

# 6. Generate SHA-256 checksums
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest_document = {
    "experiment_id": "m06_gpu_reference_unweighted_v01",
    "files": {
        weights_path.name: file_sha256(weights_path),
        history_path.name: file_sha256(history_path),
    },
    "test_status": "SEALED",
}

manifest_path.write_text(
    json.dumps(manifest_document, indent=2),
    encoding="utf-8",
)

# 7. Package model, history, and manifest into one ZIP
with zipfile.ZipFile(
    zip_path,
    mode="x",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in [weights_path, history_path, manifest_path]:
        archive.write(path, arcname=path.name)

# 8. Verify ZIP integrity
with zipfile.ZipFile(zip_path, "r") as archive:
    assert archive.testzip() is None
    assert len(archive.namelist()) == 3

print("GPU REFERENCE — ARTIFACT EXPORT")
print("=" * 60)

print("Model weights: SAVED")
print("Five-epoch history: SAVED")
print("Configuration: SAVED")
print("Checksums: SAVED")

print("\nZIP filename:", zip_path.name)
print("ZIP SHA-256:", file_sha256(zip_path))

print("\nFinal validation PR-AUC:",
      round(m06_reference_final["val_pr_auc"], 6))

print("Final validation ROC-AUC:",
      round(m06_reference_final["val_roc_auc"], 6))

print("\nWeights reload verification: PASS")
print("ZIP integrity verification: PASS")
print("Month-7 test set: SEALED")
print("Step 06.28C: PASS")

# 9. Download permanent copy to your Windows computer
files.download(str(zip_path))

**Initialize the Class-Weighted GPU Model**

In [ ]:

# ============================================================
# Milestone 06C — Cell 15: Class-Weighted GPU Model Setup
# ============================================================

import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.models.mlp import FraudMLP

# 1. Confirm required data and reference model are available
assert torch.cuda.is_available()
assert len(m06_reference_history) == 5
assert len(m06_train_dataset) == 794989
assert len(m06_validation_dataset) == 108168

# 2. Use the same settings as the unweighted reference
M06_WEIGHTED_SEED = 42
M06_WEIGHTED_EPOCHS = 5
M06_WEIGHTED_LR = 0.001
M06_WEIGHTED_BATCH_SIZE = 1024

# 3. Reset random seeds BEFORE creating the new model
random.seed(M06_WEIGHTED_SEED)
np.random.seed(M06_WEIGHTED_SEED)
torch.manual_seed(M06_WEIGHTED_SEED)
torch.cuda.manual_seed_all(M06_WEIGHTED_SEED)

m06_weighted_device = torch.device("cuda")

# 4. Create a NEW model with the original architecture
m06_weighted_model = FraudMLP(
    input_dim=55,
    dropout_rate=0.10,
).to(m06_weighted_device)

assert m06_weighted_model is not m06_reference_model

# 5. Calculate class weight from TRAINING labels only
positive_count = int(m06_y_train_tensor.sum().item())
negative_count = len(m06_y_train_tensor) - positive_count

assert positive_count == 8151
assert negative_count == 786838

m06_weighted_pos_weight = negative_count / positive_count

# 6. Initialize class-weighted BCE loss
m06_weighted_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [m06_weighted_pos_weight],
        dtype=torch.float32,
        device=m06_weighted_device,
    )
)

# 7. Create a NEW Adam optimizer
m06_weighted_optimizer = torch.optim.Adam(
    m06_weighted_model.parameters(),
    lr=M06_WEIGHTED_LR,
)

# 8. Restore the original training shuffle sequence
m06_weighted_train_loader = DataLoader(
    m06_train_dataset,
    batch_size=M06_WEIGHTED_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(
        M06_WEIGHTED_SEED
    ),
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

# Validation stays unchanged
m06_weighted_validation_loader = m06_validation_loader

# 9. Verify architecture and loaders
parameter_count = sum(
    p.numel() for p in m06_weighted_model.parameters()
)

assert parameter_count == 5697
assert len(m06_weighted_train_loader) == 777
assert len(m06_weighted_validation_loader) == 106

# 10. Test forward pass and weighted loss without training
m06_weighted_model.eval()

with torch.inference_mode():
    sample_X = torch.zeros(
        (4, 55),
        dtype=torch.float32,
        device=m06_weighted_device,
    )

    sample_y = torch.tensor(
        [0.0, 1.0, 0.0, 1.0],
        dtype=torch.float32,
        device=m06_weighted_device,
    )

    sample_logits = m06_weighted_model(sample_X)

    sample_loss = m06_weighted_criterion(
        sample_logits,
        sample_y,
    )

assert sample_logits.shape == (4,)
assert torch.isfinite(sample_loss)

m06_weighted_model.train()

# 11. Display setup results
print("CLASS-WEIGHTED GPU MODEL — SETUP")
print("=" * 55)

print("GPU:", torch.cuda.get_device_name(0))
print("Architecture: FraudMLP")
print("Trainable parameters:", parameter_count)
print("Random seed:", M06_WEIGHTED_SEED)

print("\nTRAINING CONFIGURATION")
print("=" * 55)

print("Optimizer: Adam")
print("Learning rate:", M06_WEIGHTED_LR)
print("Batch size:", M06_WEIGHTED_BATCH_SIZE)
print("Epochs:", M06_WEIGHTED_EPOCHS)
print("Positive class weight:", round(m06_weighted_pos_weight, 4))

print("\nDATA VALIDATION")
print("=" * 55)

print("Training batches:", len(m06_weighted_train_loader))
print("Validation batches:", len(m06_weighted_validation_loader))
print("Forward-pass output shape:", tuple(sample_logits.shape))

print("\nIndependent model initialization: PASS")
print("Training-only class weighting: PASS")
print("Fresh reproducible DataLoader: PASS")
print("Weighted loss smoke test: PASS")
print("Training performed: NO")
print("Month-7 test set: SEALED")
print("Step 06.29A: PASS")

**Train the Class-Weighted GPU Model**

In [ ]:

# ============================================================
# Milestone 06C — Cell 16: Train Class-Weighted GPU Model
# ============================================================

import time
import numpy as np
import torch
from torch import nn

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

# 1. Verify prerequisites before modifying model weights
assert torch.cuda.is_available()
assert len(m06_reference_history) == 5
assert len(m06_weighted_train_loader) == 777
assert len(m06_weighted_validation_loader) == 106
assert m06_weighted_criterion.pos_weight is not None

# Prevent accidental continuation of a previous training run.
if globals().get("m06_weighted_run_started", False):
    raise RuntimeError(
        "Weighted training has already started. "
        "Do not rerun this cell without reviewing model state."
    )

m06_weighted_run_started = True
m06_weighted_history = []

# Unweighted validation loss is comparable across experiments.
m06_weighted_validation_criterion = nn.BCEWithLogitsLoss()

print("CLASS-WEIGHTED GPU MODEL — TRAINING")
print("=" * 60)

print("GPU:", torch.cuda.get_device_name(0))
print("Epochs:", M06_WEIGHTED_EPOCHS)
print("Training batches:", len(m06_weighted_train_loader))
print("Validation batches:", len(m06_weighted_validation_loader))
print("Positive class weight:", round(m06_weighted_pos_weight, 4))
print("Validation loss: UNWEIGHTED for fair comparison")
print("Month-7 test set: SEALED")
print()

# 2. Train for exactly five epochs
for epoch in range(1, M06_WEIGHTED_EPOCHS + 1):

    torch.cuda.synchronize()
    epoch_start = time.perf_counter()

    # ----------------------------------------
    # TRAINING: CLASS-WEIGHTED LOSS
    # ----------------------------------------
    m06_weighted_model.train()

    training_loss_total = 0.0
    training_records = 0

    for features, labels in m06_weighted_train_loader:

        features = features.to(
            m06_weighted_device,
            non_blocking=True,
        )

        labels = labels.to(
            m06_weighted_device,
            non_blocking=True,
        )

        m06_weighted_optimizer.zero_grad(set_to_none=True)

        logits = m06_weighted_model(features)

        assert logits.shape == labels.shape

        loss = m06_weighted_criterion(logits, labels)

        loss.backward()
        m06_weighted_optimizer.step()

        batch_size = labels.shape[0]

        training_loss_total += loss.item() * batch_size
        training_records += batch_size

    assert training_records == 794989

    average_training_loss = (
        training_loss_total / training_records
    )

    # ----------------------------------------
    # VALIDATION: NO WEIGHT UPDATES
    # ----------------------------------------
    m06_weighted_model.eval()

    validation_loss_total = 0.0
    validation_records = 0

    validation_labels = []
    validation_scores = []

    with torch.inference_mode():

        for features, labels in m06_weighted_validation_loader:

            features = features.to(
                m06_weighted_device,
                non_blocking=True,
            )

            labels = labels.to(
                m06_weighted_device,
                non_blocking=True,
            )

            logits = m06_weighted_model(features)

            assert logits.shape == labels.shape

            # Unweighted validation loss
            val_loss = m06_weighted_validation_criterion(
                logits,
                labels,
            )

            batch_size = labels.shape[0]

            validation_loss_total += (
                val_loss.item() * batch_size
            )
            validation_records += batch_size

            probabilities = torch.sigmoid(logits)

            validation_labels.append(
                labels.cpu().numpy()
            )

            validation_scores.append(
                probabilities.cpu().numpy()
            )

    assert validation_records == 108168

    y_true = np.concatenate(validation_labels)
    y_score = np.concatenate(validation_scores)

    assert y_true.shape == (108168,)
    assert y_score.shape == (108168,)
    assert int(y_true.sum()) == 1450
    assert np.isfinite(y_score).all()

    average_validation_loss = (
        validation_loss_total / validation_records
    )

    validation_pr_auc = average_precision_score(
        y_true,
        y_score,
    )

    validation_roc_auc = roc_auc_score(
        y_true,
        y_score,
    )

    torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - epoch_start

    # ----------------------------------------
    # RECORD RESULTS
    # ----------------------------------------
    epoch_result = {
        "epoch": epoch,
        "train_weighted_loss": float(average_training_loss),
        "val_unweighted_loss": float(average_validation_loss),
        "val_pr_auc": float(validation_pr_auc),
        "val_roc_auc": float(validation_roc_auc),
        "epoch_seconds": float(epoch_seconds),
    }

    m06_weighted_history.append(epoch_result)

    print(
        f"Epoch {epoch}/{M06_WEIGHTED_EPOCHS} | "
        f"Weighted Train Loss: {average_training_loss:.6f} | "
        f"Unweighted Val Loss: {average_validation_loss:.6f} | "
        f"PR-AUC: {validation_pr_auc:.6f} | "
        f"ROC-AUC: {validation_roc_auc:.6f} | "
        f"Time: {epoch_seconds:.2f}s",
        flush=True,
    )

# 3. Fixed five-epoch final result
assert len(m06_weighted_history) == 5

m06_weighted_final = m06_weighted_history[-1]

m06_weighted_total_seconds = sum(
    result["epoch_seconds"]
    for result in m06_weighted_history
)

# 4. Compare against the actual GPU reference result
reference_pr_auc = m06_reference_final["val_pr_auc"]
weighted_pr_auc = m06_weighted_final["val_pr_auc"]

pr_auc_difference = weighted_pr_auc - reference_pr_auc

print("\nFINAL CLASS-WEIGHTED GPU RESULTS")
print("=" * 60)

print("Unweighted reference PR-AUC:", round(reference_pr_auc, 6))
print("Weighted model PR-AUC:", round(weighted_pr_auc, 6))
print("PR-AUC difference:", round(pr_auc_difference, 6))

print(
    "Weighted model ROC-AUC:",
    round(m06_weighted_final["val_roc_auc"], 6)
)

print(
    "Weighted model unweighted validation loss:",
    round(m06_weighted_final["val_unweighted_loss"], 6)
)

print(
    "Total training + validation time (seconds):",
    round(m06_weighted_total_seconds, 2)
)

print("\nFive-epoch training: PASS")
print("Weighted training loss: PASS")
print("Validation metrics: PASS")
print("Reference comparison: PASS")
print("Month-7 test set: SEALED")
print("Step 06.29B: PASS")

**Save the Class-Weighted GPU Model**

In [ ]:

# ============================================================
# Milestone 06C — Cell 17: Export Class-Weighted GPU Model
# ============================================================

from pathlib import Path
from google.colab import files
import torch
import json
import hashlib
import zipfile

# 1. Verify the weighted experiment completed
assert m06_weighted_run_started is True
assert len(m06_weighted_history) == 5
assert m06_weighted_final["epoch"] == 5
assert int(m06_y_train_tensor.sum().item()) == 8151

# 2. Define unique artifact filenames
export_dir = Path("/content/m06_weighted_export_v01")
zip_path = Path("/content/m06_class_weighted_gpu_v01.zip")

weights_path = export_dir / "m06_class_weighted_gpu_v01.pt"
history_path = export_dir / "m06_class_weighted_history_v01.json"
manifest_path = export_dir / "m06_class_weighted_manifest_v01.json"

# Never silently overwrite a previously saved experiment
assert not zip_path.exists(), (
    "Export ZIP already exists. Review before overwriting."
)

assert not any(
    p.exists()
    for p in [weights_path, history_path, manifest_path]
), "Existing export files detected. Review before proceeding."

export_dir.mkdir(parents=True, exist_ok=True)

# 3. Save CPU-compatible model weights
cpu_weights = {
    name: tensor.detach().cpu().clone()
    for name, tensor in m06_weighted_model.state_dict().items()
}

torch.save(cpu_weights, weights_path)

# 4. Independently reload and verify saved weights
loaded_weights = torch.load(
    weights_path,
    map_location="cpu",
    weights_only=True,
)

assert loaded_weights.keys() == cpu_weights.keys()

for name in cpu_weights:
    assert torch.equal(loaded_weights[name], cpu_weights[name])
    assert torch.isfinite(loaded_weights[name]).all()

# 5. Save configuration, provenance, and training history
history_document = {
    "experiment_id": "m06_class_weighted_gpu_v01",
    "source_commit": "cf443a5",
    "architecture": "FraudMLP",
    "feature_count": 55,
    "parameter_count": 5697,
    "seed": 42,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "batch_size": 1024,
    "epochs": 5,
    "training_loss": "BCEWithLogitsLoss_pos_weight",
    "validation_loss": "BCEWithLogitsLoss_unweighted",
    "positive_class_weight": float(m06_weighted_pos_weight),
    "train_positive_count": 8151,
    "train_negative_count": 786838,
    "train_months": [0, 1, 2, 3, 4, 5],
    "validation_months": [6],
    "test_status": "SEALED",
    "development_archive_sha256": (
        "b0924b639b9f38eceb2db413f7fa710b94dbc71d1223ab95a7db8f106be05fd5"
    ),
    "device": str(m06_weighted_device),
    "gpu": torch.cuda.get_device_name(0),
    "pytorch_version": torch.__version__,
    "epoch_history": m06_weighted_history,
    "final_validation": m06_weighted_final,
    "total_seconds": float(m06_weighted_total_seconds),
    "unweighted_reference_pr_auc": float(
        m06_reference_final["val_pr_auc"]
    ),
}

history_path.write_text(
    json.dumps(history_document, indent=2, allow_nan=False),
    encoding="utf-8",
)

# 6. Generate SHA-256 integrity checksums
def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

manifest_document = {
    "experiment_id": "m06_class_weighted_gpu_v01",
    "files": {
        weights_path.name: sha256_file(weights_path),
        history_path.name: sha256_file(history_path),
    },
    "test_status": "SEALED",
}

manifest_path.write_text(
    json.dumps(manifest_document, indent=2),
    encoding="utf-8",
)

# 7. Create the versioned ZIP archive
with zipfile.ZipFile(
    zip_path,
    mode="x",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for path in [weights_path, history_path, manifest_path]:
        archive.write(path, arcname=path.name)

# 8. Verify ZIP contents and internal checksums
with zipfile.ZipFile(zip_path, "r") as archive:

    assert archive.testzip() is None
    assert len(archive.namelist()) == 3

    for filename, expected_hash in manifest_document["files"].items():
        actual_hash = hashlib.sha256(
            archive.read(filename)
        ).hexdigest()

        assert actual_hash == expected_hash

# 9. Display export results
print("CLASS-WEIGHTED GPU MODEL — ARTIFACT EXPORT")
print("=" * 60)

print("Model weights: SAVED")
print("Five-epoch history: SAVED")
print("Experiment configuration: SAVED")
print("Integrity checksums: SAVED")

print("\nZIP filename:", zip_path.name)
print("ZIP SHA-256:", sha256_file(zip_path))

print(
    "\nFinal validation PR-AUC:",
    round(m06_weighted_final["val_pr_auc"], 6)
)

print(
    "Final validation ROC-AUC:",
    round(m06_weighted_final["val_roc_auc"], 6)
)

print("\nWeights reload verification: PASS")
print("Internal checksum verification: PASS")
print("ZIP integrity verification: PASS")
print("Month-7 test set: SEALED")
print("Step 06.29C: PASS")

# 10. Download the verified artifact to Windows
files.download(str(zip_path))

**Evaluate Fraud Detection at a 0.50 Threshold**

In [ ]:

# ============================================================
# Milestone 06C — Cell 18: Validation-Only Score Diagnostics
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
)

# 1. Confirm trained models and validation data exist
required_names = [
    "m06_reference_model",
    "m06_weighted_model",
    "m06_validation_loader",
    "m06_reference_final",
    "m06_weighted_final",
]

missing = [name for name in required_names if name not in globals()]

assert not missing, (
    f"Colab state missing: {missing}. "
    "Stop and restore the saved models before proceeding."
)

assert torch.cuda.is_available()

# 2. Evaluation only — no gradient updates
m06_reference_model.eval()
m06_weighted_model.eval()

all_labels = []
unweighted_scores = []
weighted_scores = []

with torch.inference_mode():
    for features, labels in m06_validation_loader:

        features = features.to("cuda", non_blocking=True)

        unweighted_logits = m06_reference_model(features)
        weighted_logits = m06_weighted_model(features)

        assert unweighted_logits.shape == labels.shape
        assert weighted_logits.shape == labels.shape

        all_labels.append(labels.numpy())

        unweighted_scores.append(
            torch.sigmoid(unweighted_logits).cpu().numpy()
        )

        weighted_scores.append(
            torch.sigmoid(weighted_logits).cpu().numpy()
        )

# 3. Combine predictions
y_true = np.concatenate(all_labels)

scores_by_model = {
    "Unweighted GPU": np.concatenate(unweighted_scores),
    "Class-weighted GPU": np.concatenate(weighted_scores),
}

assert y_true.shape == (108168,)
assert int(y_true.sum()) == 1450
assert np.array_equal(y_true, colab_y_validation)

# 4. Calculate threshold-based diagnostics
THRESHOLD = 0.50
diagnostic_rows = []

for model_name, scores in scores_by_model.items():

    assert scores.shape == y_true.shape
    assert np.isfinite(scores).all()
    assert np.all((scores >= 0) & (scores <= 1))

    predictions = (scores >= THRESHOLD).astype(int)

    tp = int(((predictions == 1) & (y_true == 1)).sum())
    fp = int(((predictions == 1) & (y_true == 0)).sum())
    tn = int(((predictions == 0) & (y_true == 0)).sum())
    fn = int(((predictions == 0) & (y_true == 1)).sum())

    assert tp + fp + tn + fn == len(y_true)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    diagnostic_rows.append({
        "model": model_name,
        "pr_auc": average_precision_score(y_true, scores),
        "precision_at_050": precision,
        "recall_at_050": recall,
        "false_positives": fp,
        "false_negatives": fn,
        "alert_rate": float(predictions.mean()),
        "mean_predicted_fraud_prob": float(scores.mean()),
        "brier_score": brier_score_loss(y_true, scores),
    })

# 5. Verify PR-AUC against saved training histories
m06_validation_diagnostics = pd.DataFrame(diagnostic_rows)

assert abs(
    diagnostic_rows[0]["pr_auc"]
    - m06_reference_final["val_pr_auc"]
) < 0.000005

assert abs(
    diagnostic_rows[1]["pr_auc"]
    - m06_weighted_final["val_pr_auc"]
) < 0.000005

# 6. Display comparison
print("MILESTONE 06 — VALIDATION SCORE DIAGNOSTICS")
print("=" * 60)

print("Validation records:", len(y_true))
print("Actual fraud cases:", int(y_true.sum()))
print("Threshold:", THRESHOLD)
print("Evaluation period: MONTH 6 ONLY")

display(m06_validation_diagnostics.round(6))

print("\nSaved PR-AUC reproduction: PASS")
print("Precision and recall diagnostics: PASS")
print("False-positive and false-negative counts: PASS")
print("Probability quality diagnostics: PASS")
print("Model weights updated: NO")
print("Month-7 test set: SEALED")
print("Step 06.31A: PASS")

**Evaluate Models at Equal Alert Budgets**

In [ ]:

# ============================================================
# Milestone 06C — Cell 19: Precision & Recall at Top-K
# ============================================================

import numpy as np
import pandas as pd

# 1. Confirm validation predictions are available
assert "y_true" in globals()
assert "scores_by_model" in globals()

validation_labels = np.asarray(y_true).reshape(-1)

assert validation_labels.shape == (108168,)
assert np.array_equal(
    validation_labels, colab_y_validation
)

total_fraud = int(validation_labels.sum())
assert total_fraud == 1450

# 2. Illustrative investigation capacities
ALERT_BUDGETS = [0.01, 0.05]

topk_results = []

# 3. Evaluate both models at each alert budget
for alert_budget in ALERT_BUDGETS:

    # Same investigation capacity for both models
    n_alerts = int(len(validation_labels) * alert_budget)

    for model_name, model_scores in scores_by_model.items():

        scores = np.asarray(model_scores).reshape(-1)

        assert scores.shape == validation_labels.shape
        assert np.isfinite(scores).all()

        # Rank highest fraud-risk scores first.
        # Stable sort makes tied-score handling reproducible.
        ranked_indices = np.argsort(
            -scores, kind="stable"
        )

        selected_indices = ranked_indices[:n_alerts]

        # Count actual fraud among selected transactions
        true_positives = int(
            validation_labels[selected_indices].sum()
        )

        false_positives = n_alerts - true_positives
        false_negatives = total_fraud - true_positives

        precision_at_k = true_positives / n_alerts
        recall_at_k = true_positives / total_fraud

        topk_results.append({
            "model": model_name,
            "alert_budget": f"{alert_budget:.0%}",
            "alerts": n_alerts,
            "fraud_detected": true_positives,
            "false_positives": false_positives,
            "fraud_missed": false_negatives,
            "precision_at_k": precision_at_k,
            "recall_at_k": recall_at_k,
        })

# 4. Build comparison table
m06_topk_diagnostics = pd.DataFrame(topk_results)

# 5. Verify each model received the same alert budget
for budget in ["1%", "5%"]:
    subset = m06_topk_diagnostics[
        m06_topk_diagnostics["alert_budget"] == budget
    ]

    assert len(subset) == 2
    assert subset["alerts"].nunique() == 1

assert len(m06_topk_diagnostics) == 4

# 6. Display results
print("MILESTONE 06 — TOP-K FRAUD DETECTION")
print("=" * 65)

print("Validation records:", len(validation_labels))
print("Actual fraud cases:", total_fraud)
print("Evaluation period: MONTH 6 ONLY")

display(m06_topk_diagnostics.round(6))

print("\nEqual alert-budget comparison: PASS")
print("Precision@K calculation: PASS")
print("Recall@K calculation: PASS")
print("Model weights updated: NO")
print("Month-7 test set: SEALED")
print("Step 06.31B: PASS")

**Save Validation Diagnostics**

In [ ]:

# ============================================================
# Milestone 06C — Cell 20: Export Validation Diagnostics
# ============================================================

import json
import hashlib
from pathlib import Path
from google.colab import files

# 1. Verify completed diagnostic tables
assert len(m06_validation_diagnostics) == 2
assert len(m06_topk_diagnostics) == 4
assert int(np.asarray(y_true).sum()) == 1450

# 2. Prepare a validation-only report
report = {
    "experiment_stage": "milestone_06_validation_diagnostics",
    "version": 1,
    "source_commit": "cf443a5",
    "evaluation_month": 6,
    "validation_records": 108168,
    "validation_fraud_cases": 1450,
    "classification_threshold": 0.50,
    "topk_alert_budgets": [0.01, 0.05],
    "test_status": "SEALED",
    "score_diagnostics": json.loads(
        m06_validation_diagnostics.to_json(
            orient="records",
            double_precision=15,
        )
    ),
    "topk_diagnostics": json.loads(
        m06_topk_diagnostics.to_json(
            orient="records",
            double_precision=15,
        )
    ),
}

# 3. Save without overwriting an existing report
export_path = Path(
    "/content/m06_validation_diagnostics_v01.json"
)

assert not export_path.exists(), (
    "Report already exists. Review before overwriting."
)

with export_path.open("x", encoding="utf-8") as file:
    json.dump(report, file, indent=2, allow_nan=False)

# 4. Reload and verify
restored = json.loads(
    export_path.read_text(encoding="utf-8")
)

assert restored == report
assert restored["test_status"] == "SEALED"
assert len(restored["score_diagnostics"]) == 2
assert len(restored["topk_diagnostics"]) == 4

# 5. Compute file checksum
sha256 = hashlib.sha256(
    export_path.read_bytes()
).hexdigest()

print("MILESTONE 06 — VALIDATION DIAGNOSTICS EXPORT")
print("=" * 60)

print("Filename:", export_path.name)
print("Model comparisons:", len(report["score_diagnostics"]))
print("Top-K evaluations:", len(report["topk_diagnostics"]))
print("Evaluation month:", report["evaluation_month"])
print("SHA-256:", sha256)

print("\nAggregate results saved: PASS")
print("JSON reload verification: PASS")
print("Transaction-level data exported: NO")
print("Month-7 test set: SEALED")
print("Step 06.31C: PASS")

# 6. Download report to Windows
files.download(str(export_path))

**Initialize the Intermediate-Weight GPU Model**

In [ ]:

# ============================================================
# Milestone 06C — Cell 21: Intermediate-Weight GPU Setup
# ============================================================

import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.models.mlp import FraudMLP

# 1. Verify previous experiments and development data
assert torch.cuda.is_available()
assert len(m06_reference_history) == 5
assert len(m06_weighted_history) == 5
assert len(m06_train_dataset) == 794989
assert len(m06_validation_dataset) == 108168

assert m06_y_train_tensor.shape == (794989,)
assert m06_y_validation_tensor.shape == (108168,)

# Protect an existing intermediate experiment
assert "m06_intermediate_model" not in globals(), (
    "An intermediate model already exists. "
    "Stop and inspect before reinitializing."
)

# 2. Define the controlled experiment
M06_INTERMEDIATE_SEED = 42
M06_INTERMEDIATE_EPOCHS = 5
M06_INTERMEDIATE_LR = 0.001
M06_INTERMEDIATE_BATCH_SIZE = 1024
M06_INTERMEDIATE_POS_WEIGHT = 25.0

m06_intermediate_device = torch.device("cuda")

# 3. Set the same starting random seed
random.seed(M06_INTERMEDIATE_SEED)
np.random.seed(M06_INTERMEDIATE_SEED)
torch.manual_seed(M06_INTERMEDIATE_SEED)
torch.cuda.manual_seed_all(M06_INTERMEDIATE_SEED)

# 4. Create a NEW model
m06_intermediate_model = FraudMLP(
    input_dim=55,
    dropout_rate=0.10,
).to(m06_intermediate_device)

assert m06_intermediate_model is not m06_reference_model
assert m06_intermediate_model is not m06_weighted_model

# 5. Configure intermediate class weighting
m06_intermediate_criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [M06_INTERMEDIATE_POS_WEIGHT],
        dtype=torch.float32,
        device=m06_intermediate_device,
    )
)

# Use unweighted loss for comparable validation results
m06_intermediate_validation_criterion = (
    nn.BCEWithLogitsLoss()
)

# 6. Initialize a fresh optimizer
m06_intermediate_optimizer = torch.optim.Adam(
    m06_intermediate_model.parameters(),
    lr=M06_INTERMEDIATE_LR,
)

# 7. Create a fresh, reproducibly shuffled DataLoader
m06_intermediate_train_loader = DataLoader(
    m06_train_dataset,
    batch_size=M06_INTERMEDIATE_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(
        M06_INTERMEDIATE_SEED
    ),
    num_workers=0,
    pin_memory=True,
    drop_last=False,
)

m06_intermediate_validation_loader = (
    m06_validation_loader
)

# 8. Verify architecture and batch counts
parameter_count = sum(
    p.numel()
    for p in m06_intermediate_model.parameters()
)

assert parameter_count == 5697
assert len(m06_intermediate_train_loader) == 777
assert len(m06_intermediate_validation_loader) == 106

assert (
    m06_intermediate_criterion.pos_weight.item()
    == M06_INTERMEDIATE_POS_WEIGHT
)

# 9. Smoke test: no training or weight updates
m06_intermediate_model.eval()

with torch.inference_mode():

    sample_X = torch.zeros(
        (4, 55),
        dtype=torch.float32,
        device=m06_intermediate_device,
    )

    sample_y = torch.tensor(
        [0.0, 1.0, 0.0, 1.0],
        dtype=torch.float32,
        device=m06_intermediate_device,
    )

    sample_logits = m06_intermediate_model(sample_X)

    sample_loss = m06_intermediate_criterion(
        sample_logits,
        sample_y,
    )

assert sample_logits.shape == (4,)
assert torch.isfinite(sample_loss).item()

# Restore training mode for the next step
m06_intermediate_model.train()

# 10. Display verification report
print("INTERMEDIATE-WEIGHT GPU MODEL — SETUP")
print("=" * 60)

print("GPU:", torch.cuda.get_device_name(0))
print("Architecture: FraudMLP")
print("Trainable parameters:", parameter_count)
print("Random seed:", M06_INTERMEDIATE_SEED)

print("\nTRAINING CONFIGURATION")
print("=" * 60)

print("Optimizer: Adam")
print("Learning rate:", M06_INTERMEDIATE_LR)
print("Batch size:", M06_INTERMEDIATE_BATCH_SIZE)
print("Epochs:", M06_INTERMEDIATE_EPOCHS)
print("Positive class weight:", M06_INTERMEDIATE_POS_WEIGHT)
print("Validation loss: UNWEIGHTED")

print("\nDATA AND MODEL CHECKS")
print("=" * 60)

print("Training batches:", len(m06_intermediate_train_loader))
print("Validation batches:", len(m06_intermediate_validation_loader))
print("Forward-pass shape:", tuple(sample_logits.shape))

print("\nIndependent model initialization: PASS")
print("Intermediate class weighting: PASS")
print("Fresh reproducible DataLoader: PASS")
print("Forward-pass and loss smoke test: PASS")
print("Training performed: NO")
print("Month-7 test set: SEALED")
print("Step 06.32A: PASS")

**Train the Intermediate-Weight GPU Model**

In [ ]:

# ============================================================
# Milestone 06C — Cell 22: Train Intermediate-Weight GPU MLP
# ============================================================

import time
import numpy as np
import torch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

# 1. Verify prerequisites
assert torch.cuda.is_available()
assert len(m06_intermediate_train_loader) == 777
assert len(m06_intermediate_validation_loader) == 106
assert M06_INTERMEDIATE_POS_WEIGHT == 25.0
assert m06_intermediate_criterion.pos_weight.item() == 25.0

assert len(m06_reference_history) == 5
assert len(m06_weighted_history) == 5

# Prevent accidental repeat training
if globals().get("m06_intermediate_run_started", False):
    raise RuntimeError(
        "Intermediate training has already started. "
        "Do not rerun this cell without reviewing model state."
    )

m06_intermediate_run_started = True
m06_intermediate_history = []

print("INTERMEDIATE-WEIGHT GPU MODEL — TRAINING")
print("=" * 60)

print("GPU:", torch.cuda.get_device_name(0))
print("Positive class weight:", M06_INTERMEDIATE_POS_WEIGHT)
print("Epochs:", M06_INTERMEDIATE_EPOCHS)
print("Training batches:", len(m06_intermediate_train_loader))
print("Validation batches:", len(m06_intermediate_validation_loader))
print("Training loss: WEIGHTED BCE")
print("Validation loss: UNWEIGHTED BCE")
print("Month-7 test set: SEALED")
print()

# 2. Train and validate for five epochs
for epoch in range(1, M06_INTERMEDIATE_EPOCHS + 1):

    torch.cuda.synchronize()
    epoch_start = time.perf_counter()

    # ---------------------------
    # TRAINING
    # ---------------------------
    m06_intermediate_model.train()

    train_loss_total = 0.0
    train_records = 0

    for features, labels in m06_intermediate_train_loader:

        features = features.to(
            m06_intermediate_device,
            non_blocking=True,
        )

        labels = labels.to(
            m06_intermediate_device,
            non_blocking=True,
        )

        m06_intermediate_optimizer.zero_grad(
            set_to_none=True
        )

        logits = m06_intermediate_model(features)

        assert logits.shape == labels.shape

        loss = m06_intermediate_criterion(
            logits,
            labels,
        )

        loss.backward()
        m06_intermediate_optimizer.step()

        batch_size = labels.shape[0]

        train_loss_total += loss.item() * batch_size
        train_records += batch_size

    assert train_records == 794989

    avg_train_loss = train_loss_total / train_records

    # ---------------------------
    # VALIDATION — NO TRAINING
    # ---------------------------
    m06_intermediate_model.eval()

    val_loss_total = 0.0
    val_records = 0

    all_labels = []
    all_scores = []

    with torch.inference_mode():

        for features, labels in m06_intermediate_validation_loader:

            features = features.to(
                m06_intermediate_device,
                non_blocking=True,
            )

            labels = labels.to(
                m06_intermediate_device,
                non_blocking=True,
            )

            logits = m06_intermediate_model(features)

            assert logits.shape == labels.shape

            # Unweighted validation loss
            val_loss = m06_intermediate_validation_criterion(
                logits,
                labels,
            )

            batch_size = labels.shape[0]

            val_loss_total += val_loss.item() * batch_size
            val_records += batch_size

            probabilities = torch.sigmoid(logits)

            all_labels.append(
                labels.cpu().numpy()
            )

            all_scores.append(
                probabilities.cpu().numpy()
            )

    assert val_records == 108168

    y_true_intermediate = np.concatenate(all_labels)
    y_score_intermediate = np.concatenate(all_scores)

    assert y_true_intermediate.shape == (108168,)
    assert y_score_intermediate.shape == (108168,)
    assert int(y_true_intermediate.sum()) == 1450
    assert np.array_equal(
        y_true_intermediate,
        colab_y_validation,
    )
    assert np.isfinite(y_score_intermediate).all()

    avg_val_loss = val_loss_total / val_records

    val_pr_auc = average_precision_score(
        y_true_intermediate,
        y_score_intermediate,
    )

    val_roc_auc = roc_auc_score(
        y_true_intermediate,
        y_score_intermediate,
    )

    torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - epoch_start

    # ---------------------------
    # SAVE EPOCH RESULTS IN MEMORY
    # ---------------------------
    epoch_result = {
        "epoch": epoch,
        "train_weighted_loss": float(avg_train_loss),
        "val_unweighted_loss": float(avg_val_loss),
        "val_pr_auc": float(val_pr_auc),
        "val_roc_auc": float(val_roc_auc),
        "epoch_seconds": float(epoch_seconds),
    }

    m06_intermediate_history.append(epoch_result)

    print(
        f"Epoch {epoch}/{M06_INTERMEDIATE_EPOCHS} | "
        f"Weighted Train Loss: {avg_train_loss:.6f} | "
        f"Unweighted Val Loss: {avg_val_loss:.6f} | "
        f"PR-AUC: {val_pr_auc:.6f} | "
        f"ROC-AUC: {val_roc_auc:.6f} | "
        f"Time: {epoch_seconds:.2f}s",
        flush=True,
    )

# 3. Fixed five-epoch result
assert len(m06_intermediate_history) == 5

m06_intermediate_final = m06_intermediate_history[-1]

m06_intermediate_total_seconds = sum(
    result["epoch_seconds"]
    for result in m06_intermediate_history
)

# 4. Compare all three GPU experiments
reference_pr_auc = m06_reference_final["val_pr_auc"]
full_weight_pr_auc = m06_weighted_final["val_pr_auc"]
intermediate_pr_auc = m06_intermediate_final["val_pr_auc"]

print("\nFINAL GPU MODEL COMPARISON")
print("=" * 60)

print("Unweighted PR-AUC:", round(reference_pr_auc, 6))
print("Full-weight PR-AUC:", round(full_weight_pr_auc, 6))
print("Weight-25 PR-AUC:", round(intermediate_pr_auc, 6))

print(
    "\nWeight-25 minus unweighted:",
    round(intermediate_pr_auc - reference_pr_auc, 6)
)

print(
    "Weight-25 minus full-weight:",
    round(intermediate_pr_auc - full_weight_pr_auc, 6)
)

print(
    "\nWeight-25 ROC-AUC:",
    round(m06_intermediate_final["val_roc_auc"], 6)
)

print(
    "Weight-25 unweighted validation loss:",
    round(m06_intermediate_final["val_unweighted_loss"], 6)
)

print(
    "Total training + validation time (seconds):",
    round(m06_intermediate_total_seconds, 2)
)

print("\nFive-epoch training: PASS")
print("Intermediate-weight loss: PASS")
print("Validation metrics: PASS")
print("Three-model comparison: PASS")
print("Month-7 test set: SEALED")
print("Step 06.32B: PASS")

**Three important findings**
1. Weight 96.53 remains our best neural-network configuration by fixed-epoch PR-AUC, at 0.163862.
2. Weight 25 reduces validation loss by approximately 58.6% compared with full weighting, suggesting less severe probability distortion. We still need calibration diagnostics to confirm its probability quality.
3. Weight 25 performed better at epoch 4 than epoch 5. Its epoch-4 PR-AUC was 0.163282, but we will preserve epoch 5 as the official result because our experiments use a fixed five-epoch comparison.
Our HistGradientBoosting benchmark still leads at 0.166961 PR-AUC.

**Export the Weight-25 GPU Model**

In [ ]:

# ============================================================
# Milestone 06C — Cell 23: Export Weight-25 GPU Model
# ============================================================

from pathlib import Path
from google.colab import files
import subprocess
import torch
import json
import hashlib
import zipfile

# 1. Confirm successful training
assert m06_intermediate_run_started is True
assert len(m06_intermediate_history) == 5
assert m06_intermediate_final["epoch"] == 5
assert M06_INTERMEDIATE_POS_WEIGHT == 25.0

# 2. Verify source revision
repo_path = Path("/content/bank-account-fraud-dl-mlops")

source_commit = subprocess.check_output(
    ["git", "-C", str(repo_path), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

assert source_commit == "cf443a5"

# 3. Define unique artifact paths
export_dir = Path("/content/m06_weight25_export_v01")
zip_path = Path("/content/m06_weight25_gpu_v01.zip")

weights_path = export_dir / "m06_weight25_gpu_v01.pt"
history_path = export_dir / "m06_weight25_history_v01.json"
manifest_path = export_dir / "m06_weight25_manifest_v01.json"

# Prevent accidental overwriting
assert not zip_path.exists(), "ZIP already exists."

assert not any(
    p.exists()
    for p in [weights_path, history_path, manifest_path]
), "Existing export detected."

export_dir.mkdir(parents=True, exist_ok=True)

# 4. Save portable CPU model weights
cpu_weights = {
    name: tensor.detach().cpu().clone()
    for name, tensor in m06_intermediate_model.state_dict().items()
}

torch.save(cpu_weights, weights_path)

# 5. Reload and verify every weight tensor
reloaded_weights = torch.load(
    weights_path,
    map_location="cpu",
    weights_only=True,
)

assert reloaded_weights.keys() == cpu_weights.keys()

for name in cpu_weights:
    assert torch.equal(
        reloaded_weights[name],
        cpu_weights[name],
    )
    assert torch.isfinite(reloaded_weights[name]).all()

# 6. Record configuration, metrics, and provenance
history_document = {
    "experiment_id": "m06_weight25_gpu_v01",
    "source_commit": source_commit,
    "architecture": "FraudMLP",
    "feature_count": 55,
    "parameter_count": 5697,
    "seed": 42,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "batch_size": 1024,
    "epochs": 5,
    "training_loss": "BCEWithLogitsLoss_pos_weight",
    "validation_loss": "BCEWithLogitsLoss_unweighted",
    "positive_class_weight": 25.0,
    "train_positive_count": 8151,
    "train_negative_count": 786838,
    "train_months": [0, 1, 2, 3, 4, 5],
    "validation_months": [6],
    "test_status": "SEALED",
    "development_archive_sha256": (
        "b0924b639b9f38eceb2db413f7fa710b94dbc71d1223ab95a7db8f106be05fd5"
    ),
    "device": str(m06_intermediate_device),
    "gpu": torch.cuda.get_device_name(0),
    "pytorch_version": torch.__version__,
    "epoch_history": m06_intermediate_history,
    "final_validation": m06_intermediate_final,
    "total_seconds": float(m06_intermediate_total_seconds),
    "unweighted_reference_pr_auc": float(
        m06_reference_final["val_pr_auc"]
    ),
    "full_weight_reference_pr_auc": float(
        m06_weighted_final["val_pr_auc"]
    ),
}

history_path.write_text(
    json.dumps(history_document, indent=2, allow_nan=False),
    encoding="utf-8",
)

# 7. Calculate SHA-256 checksums
def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

manifest_document = {
    "experiment_id": "m06_weight25_gpu_v01",
    "files": {
        weights_path.name: sha256_file(weights_path),
        history_path.name: sha256_file(history_path),
    },
    "test_status": "SEALED",
}

manifest_path.write_text(
    json.dumps(manifest_document, indent=2),
    encoding="utf-8",
)

# 8. Package all three files
with zipfile.ZipFile(
    zip_path,
    mode="x",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in [weights_path, history_path, manifest_path]:
        archive.write(path, arcname=path.name)

# 9. Verify ZIP and internal checksums
with zipfile.ZipFile(zip_path, "r") as archive:
    assert archive.testzip() is None
    assert len(archive.namelist()) == 3

    for filename, expected_hash in manifest_document["files"].items():
        actual_hash = hashlib.sha256(
            archive.read(filename)
        ).hexdigest()

        assert actual_hash == expected_hash

# 10. Report successful export
print("WEIGHT-25 GPU MODEL — ARTIFACT EXPORT")
print("=" * 60)

print("Model weights: SAVED")
print("Five-epoch history: SAVED")
print("Experiment configuration: SAVED")
print("Source revision: VERIFIED")
print("Integrity checksums: SAVED")

print("\nZIP filename:", zip_path.name)
print("ZIP SHA-256:", sha256_file(zip_path))

print(
    "\nValidation PR-AUC:",
    round(m06_intermediate_final["val_pr_auc"], 6)
)

print(
    "Validation ROC-AUC:",
    round(m06_intermediate_final["val_roc_auc"], 6)
)

print(
    "Unweighted validation loss:",
    round(m06_intermediate_final["val_unweighted_loss"], 6)
)

print("\nWeights reload verification: PASS")
print("Internal checksums: PASS")
print("ZIP integrity: PASS")
print("Month-7 test set: SEALED")
print("Step 06.32C: PASS")

# 11. Download to Windows
files.download(str(zip_path))

**Evaluate Weight-25 Fraud Detection**

In [ ]:

# ============================================================
# Milestone 06C — Cell 24: Weight-25 Validation Diagnostics
# ============================================================

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

# 1. Confirm all required components are available
assert torch.cuda.is_available()
assert len(m06_validation_loader) == 106
assert len(m06_intermediate_history) == 5
assert m06_intermediate_final["epoch"] == 5

assert len(m06_validation_diagnostics) == 2
assert len(m06_topk_diagnostics) == 4

# 2. Generate weight-25 predictions on MONTH 6 ONLY
m06_intermediate_model.eval()

validation_labels = []
validation_scores = []

with torch.inference_mode():

    for features, labels in m06_validation_loader:

        features = features.to(
            m06_intermediate_device,
            non_blocking=True,
        )

        logits = m06_intermediate_model(features)

        assert logits.shape == labels.shape

        scores = torch.sigmoid(logits)

        validation_labels.append(
            labels.cpu().numpy()
        )

        validation_scores.append(
            scores.cpu().numpy()
        )

# 3. Combine and verify predictions
m06_weight25_y = np.concatenate(validation_labels)

m06_weight25_scores = np.concatenate(
    validation_scores
)

assert m06_weight25_y.shape == (108168,)
assert m06_weight25_scores.shape == (108168,)

assert np.array_equal(
    m06_weight25_y,
    colab_y_validation,
)

assert int(m06_weight25_y.sum()) == 1450
assert np.isfinite(m06_weight25_scores).all()

assert np.all(
    (m06_weight25_scores >= 0)
    & (m06_weight25_scores <= 1)
)

# 4. Reproduce the saved validation metrics
pr_auc = average_precision_score(
    m06_weight25_y,
    m06_weight25_scores,
)

roc_auc = roc_auc_score(
    m06_weight25_y,
    m06_weight25_scores,
)

brier = brier_score_loss(
    m06_weight25_y,
    m06_weight25_scores,
)

assert abs(
    pr_auc - m06_intermediate_final["val_pr_auc"]
) < 0.000005

assert abs(
    roc_auc - m06_intermediate_final["val_roc_auc"]
) < 0.000005

# 5. Evaluate the 0.50 classification threshold
THRESHOLD = 0.50

predictions = (
    m06_weight25_scores >= THRESHOLD
).astype(int)

tp = int(((predictions == 1) & (m06_weight25_y == 1)).sum())
fp = int(((predictions == 1) & (m06_weight25_y == 0)).sum())
tn = int(((predictions == 0) & (m06_weight25_y == 0)).sum())
fn = int(((predictions == 0) & (m06_weight25_y == 1)).sum())

assert tp + fp + tn + fn == 108168

m06_weight25_diagnostics = {
    "model": "Weight-25 GPU",
    "pr_auc": float(pr_auc),
    "roc_auc": float(roc_auc),
    "precision_at_050": tp / (tp + fp) if tp + fp else 0.0,
    "recall_at_050": tp / (tp + fn),
    "true_positives": tp,
    "false_positives": fp,
    "false_negatives": fn,
    "alert_rate": float(predictions.mean()),
    "mean_predicted_fraud_prob": float(
        m06_weight25_scores.mean()
    ),
    "brier_score": float(brier),
}

# 6. Evaluate the SAME 1% and 5% alert budgets
ranked_indices = np.argsort(
    -m06_weight25_scores,
    kind="stable",
)

topk_rows = []

for budget in [0.01, 0.05]:

    n_alerts = int(len(m06_weight25_y) * budget)

    selected = ranked_indices[:n_alerts]

    fraud_detected = int(
        m06_weight25_y[selected].sum()
    )

    topk_rows.append({
        "model": "Weight-25 GPU",
        "alert_budget": f"{budget:.0%}",
        "alerts": n_alerts,
        "fraud_detected": fraud_detected,
        "false_positives": n_alerts - fraud_detected,
        "fraud_missed": 1450 - fraud_detected,
        "precision_at_k": fraud_detected / n_alerts,
        "recall_at_k": fraud_detected / 1450,
    })

m06_weight25_topk = pd.DataFrame(topk_rows)

# 7. Combine with previous Top-K results
m06_three_model_topk = pd.concat(
    [
        m06_topk_diagnostics,
        m06_weight25_topk,
    ],
    ignore_index=True,
)

assert len(m06_three_model_topk) == 6

# 8. Display results
print("WEIGHT-25 — VALIDATION DIAGNOSTICS")
print("=" * 60)

print("Validation records:", len(m06_weight25_y))
print("Actual fraud cases:", int(m06_weight25_y.sum()))
print("Evaluation period: MONTH 6 ONLY")

print("\nWEIGHT-25 PERFORMANCE AT THRESHOLD 0.50")

display(
    pd.DataFrame([m06_weight25_diagnostics]).round(6)
)

print("\nALL THREE MODELS — EQUAL ALERT BUDGETS")

display(
    m06_three_model_topk.sort_values(
        ["alert_budget", "model"]
    ).round(6)
)

print("\nSaved PR-AUC reproduction: PASS")
print("Saved ROC-AUC reproduction: PASS")
print("Threshold diagnostics: PASS")
print("Three-model Top-K comparison: PASS")
print("Model weights updated: NO")
print("Month-7 test set: SEALED")
print("Step 06.33A: PASS")

**Save the Three-Model Validation Report**

In [ ]:

# ============================================================
# Milestone 06C — Cell 25: Export Three-Model Diagnostics
# ============================================================

import json
import hashlib
from pathlib import Path
from google.colab import files

# 1. Verify the completed validation analyses
assert len(m06_validation_diagnostics) == 2
assert len(m06_three_model_topk) == 6
assert len(m06_weight25_topk) == 2

assert m06_weight25_diagnostics["model"] == "Weight-25 GPU"

assert abs(
    m06_weight25_diagnostics["pr_auc"]
    - m06_intermediate_final["val_pr_auc"]
) < 0.000005

# 2. Convert DataFrames into JSON-compatible records
previous_metrics = json.loads(
    m06_validation_diagnostics.to_json(
        orient="records",
        double_precision=15,
    )
)

topk_metrics = json.loads(
    m06_three_model_topk.to_json(
        orient="records",
        double_precision=15,
    )
)

# 3. Create aggregate-only report
report = {
    "report_id": "m06_three_model_diagnostics_v02",
    "version": 2,
    "source_commit": "cf443a5",
    "training_months": [0, 1, 2, 3, 4, 5],
    "validation_month": 6,
    "validation_records": 108168,
    "validation_fraud_cases": 1450,
    "test_status": "SEALED",
    "primary_metric": "validation_pr_auc",
    "classification_threshold": 0.50,
    "alert_budgets": [0.01, 0.05],
    "previous_model_diagnostics": previous_metrics,
    "weight25_diagnostics": m06_weight25_diagnostics,
    "three_model_topk": topk_metrics,
    "experiment_pr_auc": {
        "unweighted": float(
            m06_reference_final["val_pr_auc"]
        ),
        "weight25": float(
            m06_intermediate_final["val_pr_auc"]
        ),
        "full_weight": float(
            m06_weighted_final["val_pr_auc"]
        ),
    },
}

# 4. Prevent accidental overwriting
export_path = Path(
    "/content/m06_three_model_diagnostics_v02.json"
)

assert not export_path.exists(), (
    "Report already exists. Stop before overwriting."
)

# 5. Save JSON
with export_path.open("x", encoding="utf-8") as file:
    json.dump(
        report,
        file,
        indent=2,
        allow_nan=False,
    )

# 6. Reload and verify
restored = json.loads(
    export_path.read_text(encoding="utf-8")
)

assert restored == report
assert restored["test_status"] == "SEALED"
assert len(restored["previous_model_diagnostics"]) == 2
assert len(restored["three_model_topk"]) == 6

# 7. Calculate SHA-256
sha256 = hashlib.sha256(
    export_path.read_bytes()
).hexdigest()

# 8. Report results
print("MILESTONE 06 — THREE-MODEL DIAGNOSTICS EXPORT")
print("=" * 60)

print("Filename:", export_path.name)
print("GPU models evaluated: 3")
print("Top-K comparisons:", len(topk_metrics))
print("Validation month:", report["validation_month"])
print("SHA-256:", sha256)

print("\nAggregate diagnostics saved: PASS")
print("JSON reload verification: PASS")
print("Transaction-level data exported: NO")
print("Month-7 test set: SEALED")
print("Step 06.33B: PASS")

# 9. Download to Windows
files.download(str(export_path))